# Late Fusion ML Pipeline — ECG + PPG

Two separate classifiers trained on independent datasets, combined
via rule-based fusion logic. Architecture matches the embedded system:
ECG features → ECG model → ECG condition, PPG features → PPG model
→ PPG condition, both fed into the final decision layer.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    ConfusionMatrixDisplay
)
from collections import Counter

## Stage 1 — Load and Clean Data

In [2]:
ecg = pd.read_csv('all_ecg_features.csv')
print('ECG shape:', ecg.shape)
print(ecg.head())
print('\nECG nulls:\n', ecg.isnull().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'all_ecg_features.csv'

In [ ]:
ppg = pd.read_csv('ppg_ml_dataset.csv')
print('PPG shape:', ppg.shape)
print(ppg.head())
print('\nPPG nulls:\n', ppg.isnull().sum())

In [ ]:
ecg = ecg.drop_duplicates()
ecg = ecg[
    (ecg['HR_bpm']     > 0)   &
    (ecg['HR_bpm']     < 250) &
    (ecg['RR_mean_ms'] > 0)   &
    (ecg['SDNN_ms']    >= 0)  &
    (ecg['RMSSD_ms']   >= 0)
].reset_index(drop=True)

ECG_FEATURES = ['RR_mean_ms', 'HR_bpm', 'SDNN_ms',
                'RMSSD_ms', 'mean_QRS_ms', 'SQI_q10']

print('ECG after cleaning:', ecg.shape)
print(ecg[ECG_FEATURES].describe())

In [ ]:
ppg = ppg.drop_duplicates()
ppg = ppg[
    (ppg['HeartRate']      > 0)   &
    (ppg['HeartRate']      < 200) &
    (ppg['SpO2']           > 0)   &
    (ppg['SpO2']           <= 100) &
    (ppg['PerfusionIndex'] > 0)
].reset_index(drop=True)

PPG_FEATURES = ['HeartRate', 'SpO2', 'RespRate',
                'PerfusionIndex', 'RedAC', 'IRAC']

print('PPG after cleaning:', ppg.shape)
print(ppg[PPG_FEATURES].describe())

## Stage 2 — Label Generation

ECG labels use clinical HR thresholds plus HRV and signal quality checks.

PPG labels use dataset-relative SpO2 and PI thresholds (25th percentile)
because the empirical SpO2 formula produces systematically lower absolute
values than a calibrated pulse oximeter. Using the fixed clinical threshold
of 92% would label 93% of subjects as hypoxic and destroy the classifier.

In [ ]:
def ecg_label(row):
    if row['HR_bpm'] > 100:
        return 'Tachycardia'
    elif row['HR_bpm'] < 60:
        return 'Bradycardia'
    elif row['SDNN_ms'] < 20:
        return 'Low_HRV'
    elif row['SQI_q10'] < 800:
        return 'Noisy'
    else:
        return 'Normal'

ecg['label'] = ecg.apply(ecg_label, axis=1)

print('ECG label distribution:')
print(ecg['label'].value_counts())

ecg['label'].value_counts().plot(kind='bar', title='ECG Labels', color='steelblue')
plt.tight_layout()
plt.show()

In [ ]:
spo2_threshold = ppg['SpO2'].quantile(0.25)
pi_threshold   = ppg['PerfusionIndex'].quantile(0.25)

print(f'SpO2 25th percentile threshold: {spo2_threshold:.1f}')
print(f'PI   25th percentile threshold: {pi_threshold:.1f}')

def ppg_label(row):
    if row['HeartRate'] > 100:
        return 'Tachycardia'
    elif row['HeartRate'] < 60:
        return 'Bradycardia'
    elif row['SpO2'] < spo2_threshold:
        return 'Low_SpO2'
    elif row['PerfusionIndex'] < pi_threshold:
        return 'Low_Perfusion'
    else:
        return 'Normal'

ppg['label'] = ppg.apply(ppg_label, axis=1)

print('\nPPG label distribution:')
print(ppg['label'].value_counts())

ppg['label'].value_counts().plot(kind='bar', title='PPG Labels', color='darkorange')
plt.tight_layout()
plt.show()

## Stage 3 — Feature Preparation and Normalization

In [ ]:
X_ecg = ecg[ECG_FEATURES].values
y_ecg = ecg['label'].values

scaler_ecg   = StandardScaler()
X_ecg_scaled = scaler_ecg.fit_transform(X_ecg)

print('ECG feature matrix:', X_ecg_scaled.shape)
print('ECG classes:', np.unique(y_ecg))

In [ ]:
X_ppg = ppg[PPG_FEATURES].values
y_ppg = ppg['label'].values

scaler_ppg   = StandardScaler()
X_ppg_scaled = scaler_ppg.fit_transform(X_ppg)

print('PPG feature matrix:', X_ppg_scaled.shape)
print('PPG classes:', np.unique(y_ppg))

## Stage 4 — Model Training

In [ ]:
X_ecg_train, X_ecg_test, y_ecg_train, y_ecg_test = train_test_split(
    X_ecg_scaled, y_ecg, test_size=0.2, random_state=42, stratify=y_ecg
)

ecg_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
ecg_model.fit(X_ecg_train, y_ecg_train)

cv_scores    = cross_val_score(ecg_model, X_ecg_scaled, y_ecg, cv=5)
y_ecg_pred   = ecg_model.predict(X_ecg_test)

print('ECG MODEL RESULTS')
print('=' * 40)
print(f'Test accuracy : {accuracy_score(y_ecg_test, y_ecg_pred):.3f}')
print(f'CV accuracy   : {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')
print()
print(classification_report(y_ecg_test, y_ecg_pred))

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_ecg_test, y_ecg_pred, ax=ax, colorbar=False
)
ax.set_title('ECG Model — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
X_ppg_train, X_ppg_test, y_ppg_train, y_ppg_test = train_test_split(
    X_ppg_scaled, y_ppg, test_size=0.2, random_state=42, stratify=y_ppg
)

ppg_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
ppg_model.fit(X_ppg_train, y_ppg_train)

cv_scores_ppg = cross_val_score(ppg_model, X_ppg_scaled, y_ppg, cv=5)
y_ppg_pred    = ppg_model.predict(X_ppg_test)

print('PPG MODEL RESULTS')
print('=' * 40)
print(f'Test accuracy : {accuracy_score(y_ppg_test, y_ppg_pred):.3f}')
print(f'CV accuracy   : {cv_scores_ppg.mean():.3f} +/- {cv_scores_ppg.std():.3f}')
print()
print(classification_report(y_ppg_test, y_ppg_pred))

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_ppg_test, y_ppg_pred, ax=ax, colorbar=False
)
ax.set_title('PPG Model — Confusion Matrix')
plt.tight_layout()
plt.show()

## Stage 5 — Decision Tree Rules

These rules map directly to the embedded C if-else code.

In [ ]:
print('ECG DECISION TREE RULES')
print('=' * 50)
print(export_text(ecg_model, feature_names=ECG_FEATURES))

print()
print('PPG DECISION TREE RULES')
print('=' * 50)
print(export_text(ppg_model, feature_names=PPG_FEATURES))

## Stage 6 — Late Fusion Inference

In [ ]:
def final_decision(ecg_result, ppg_result):
    ecg_abnormal = ecg_result != 'Normal'
    ppg_abnormal = ppg_result != 'Normal'

    if ecg_abnormal and ppg_result == 'Low_SpO2':
        return 'CRITICAL'
    if ecg_result == 'Tachycardia' and ppg_result == 'Tachycardia':
        return 'CRITICAL'
    if ecg_result == 'Bradycardia' and ppg_result == 'Bradycardia':
        return 'CRITICAL'
    if ecg_abnormal and ppg_abnormal:
        return 'DUAL_ALERT'
    if ecg_result in ('Tachycardia', 'Bradycardia'):
        return 'CARDIAC_ALERT'
    if ecg_result == 'Low_HRV':
        return 'HRV_ALERT'
    if ppg_result == 'Low_SpO2':
        return 'OXYGEN_ALERT'
    if ppg_result == 'Low_Perfusion':
        return 'CIRCULATION_ALERT'
    if ppg_result in ('Tachycardia', 'Bradycardia'):
        return 'CARDIAC_ALERT'
    if ecg_result == 'Noisy' or ppg_abnormal:
        return 'CHECK_SIGNAL'
    return 'NORMAL'


ecg_preds_all = ecg_model.predict(X_ecg_scaled)
ppg_preds_all = ppg_model.predict(X_ppg_scaled)
n_pairs       = min(len(ecg_preds_all), len(ppg_preds_all))

fusion_results = [
    final_decision(ecg_preds_all[i], ppg_preds_all[i])
    for i in range(n_pairs)
]

print('FUSION OUTPUT DISTRIBUTION')
print('=' * 40)
for decision, count in Counter(fusion_results).most_common():
    print(f'  {decision:<22}: {count}')

In [ ]:
print('SINGLE PATIENT DEMO')
print('=' * 40)

ecg_sample = X_ecg_scaled[0].reshape(1, -1)
ppg_sample = X_ppg_scaled[0].reshape(1, -1)

ecg_result = ecg_model.predict(ecg_sample)[0]
ppg_result = ppg_model.predict(ppg_sample)[0]
decision   = final_decision(ecg_result, ppg_result)

print(f'ECG raw features : {dict(zip(ECG_FEATURES, X_ecg[0]))}')
print(f'PPG raw features : {dict(zip(PPG_FEATURES, X_ppg[0]))}')
print(f'ECG model output : {ecg_result}')
print(f'PPG model output : {ppg_result}')
print(f'FINAL DECISION   : {decision}')

## Stage 7 — Export Decision Tree as C Code

In [ ]:
def tree_to_c(model, feature_names, function_name):
    tree = model.tree_
    classes = model.classes_
    lines = []
    lines.append('/* Auto-generated from DecisionTreeClassifier */')
    lines.append(f'/* Features: {", ".join(feature_names)} */')
    lines.append('')
    args = ', '.join(f'float {f}' for f in feature_names)
    lines.append(f'const char* {function_name}({args})')
    lines.append('{')

    def recurse(node, depth):
        indent = '    ' * (depth + 1)
        if tree.feature[node] != -2:
            feat   = feature_names[tree.feature[node]]
            thresh = tree.threshold[node]
            lines.append(f'{indent}if ({feat} <= {thresh:.6f}f) {{')
            recurse(tree.children_left[node],  depth + 1)
            lines.append(f'{indent}}} else {{')
            recurse(tree.children_right[node], depth + 1)
            lines.append(f'{indent}}}')
        else:
            label = classes[np.argmax(tree.value[node])]
            lines.append(f'{indent}return "{label}";')

    recurse(0, 0)
    lines.append('}')
    return '\n'.join(lines)


ecg_c = tree_to_c(ecg_model, ECG_FEATURES, 'ecg_classify')
ppg_c = tree_to_c(ppg_model, PPG_FEATURES, 'ppg_classify')

with open('ecg_model.c', 'w') as f:
    f.write(ecg_c)

with open('ppg_model.c', 'w') as f:
    f.write(ppg_c)

print('ecg_model.c and ppg_model.c saved')
print()
print(ecg_c[:600])
print('...')

In [ ]:
fusion_c = '''/* Late fusion decision logic */
#include <string.h>

typedef enum {
    DECISION_NORMAL,
    DECISION_CARDIAC_ALERT,
    DECISION_HRV_ALERT,
    DECISION_OXYGEN_ALERT,
    DECISION_CIRCULATION_ALERT,
    DECISION_DUAL_ALERT,
    DECISION_CRITICAL,
    DECISION_CHECK_SIGNAL
} FusionDecision;

FusionDecision fusion_decide(const char *ecg, const char *ppg)
{
    int ea = strcmp(ecg, "Normal") != 0;
    int pa = strcmp(ppg, "Normal") != 0;

    if (ea && strcmp(ppg, "Low_SpO2") == 0)      return DECISION_CRITICAL;
    if (strcmp(ecg,"Tachycardia")==0 && strcmp(ppg,"Tachycardia")==0) return DECISION_CRITICAL;
    if (strcmp(ecg,"Bradycardia")==0 && strcmp(ppg,"Bradycardia")==0) return DECISION_CRITICAL;
    if (ea && pa)                                 return DECISION_DUAL_ALERT;
    if (strcmp(ecg,"Tachycardia")==0 || strcmp(ecg,"Bradycardia")==0) return DECISION_CARDIAC_ALERT;
    if (strcmp(ecg,"Low_HRV")==0)                return DECISION_HRV_ALERT;
    if (strcmp(ppg,"Low_SpO2")==0)               return DECISION_OXYGEN_ALERT;
    if (strcmp(ppg,"Low_Perfusion")==0)          return DECISION_CIRCULATION_ALERT;
    if (strcmp(ppg,"Tachycardia")==0 || strcmp(ppg,"Bradycardia")==0) return DECISION_CARDIAC_ALERT;
    if (strcmp(ecg,"Noisy")==0 || pa)            return DECISION_CHECK_SIGNAL;
    return DECISION_NORMAL;
}
'''

with open('fusion_logic.c', 'w') as f:
    f.write(fusion_c)

print('fusion_logic.c saved')
print(fusion_c)